# Test code for api

In [17]:
import os
import pandas as pd
import re
from CM.utils import *
from CM.keys import *

import zipfile
import numpy as np

In [18]:
database = "SocioMap"
syntax = "R"
dirpath = "./tmp"
template = pd.read_csv("55-template.csv")
driver = getDriver(database)

In [19]:
def transform_variables_r(variables):
    variables["transform"] = variables["transform"].str.replace(
        "~", "!", regex=True)
    variables["transform"] = variables["transform"].str.replace(
        "=", "==", regex=True)
    variables["transform"] = variables["transform"].str.replace(
        "!==", "!=", regex=True)
    variables["transform"] = variables["transform"].str.replace(
        "concat", "paste0", regex=True)
    variables["transform"] = variables["transform"].str.replace(
        r',0\)', ',na.rm = True', regex=True)
    variables["transform"] = variables["transform"].str.replace(
        "in", "%in%", regex=True)
    variables["transform"] = variables["transform"].str.replace(
        "na.rm == T", "na.rm = True", regex=True)
    variables["transform"] = variables["transform"].str.replace(
        "== as.numeric", "= as.numeric", regex=True)
    return variables

In [20]:
def load_r_syntax_template(filename, replacements):
    """ Reads R syntax template and replaces placeholders """
    try:
        with open(filename, "r") as file:
            content = file.read()
        for key, value in replacements.items():
            content = content.replace(key, value)
        return content
    except FileNotFoundError:
        print(f"Error: {filename} not found. Ensure the file exists.")
        return None

In [21]:
def zip_output_files(dirpath, zip_filename="output.zip"):
    """ Zip all generated files into a single archive """
    zip_path = os.path.join(dirpath, zip_filename)

    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for file in files:
            if os.path.exists(file):  # Ensure file exists before adding
                # Store without full path
                zipf.write(file, os.path.basename(file))
                print(f"Added to ZIP: {file}")
            else:
                print(
                    f"Warning: {file} does not exist and was not added to the ZIP.")

    print(f"ZIP file created: {zip_path}")
    return zip_path

In [22]:
if "filePath" not in template.columns:
    raise ValueError(
        "Must upload a list of datasets with the filePath column before generating syntax.")

wd = template.iloc[0]["filePath"]

if re.match(r"^[a-zA-Z]:\\\\", wd) or "\\" in wd:
    print("Detected Windows path. Converting to compatible format...")

    # Convert backslashes to forward slashes
    wd = wd.replace("\\", "/")

    # Ensure proper formatting (R escape sequences)
    wd = wd.replace(" ", "\\ ")  # Escape spaces if needed

template = template.iloc[1:]

# verify CMIDs
cols = ["mergingID", "stackID", "datasetID"]
cols = [col for col in cols if col in template.columns]
CMIDs = list(set(template[cols].values.flatten().tolist()))

check = getQuery(
    """
    UNWIND $CMIDs as cmid
    match (a:DATASET {CMID: cmid})
    return a.CMID as CMID, a.CMName as CMName
    """, driver = driver,
    params={"CMIDs": CMIDs},
    type = "df"
)

missing = set(CMIDs) - set(check["CMID"].tolist())
missing = [str(m) + "\n" for m in missing]

if len(check) != len(CMIDs):
    raise ValueError(
        "Error: One or more CMIDs not found in the database\nMissing CMIDs: ", missing)
else:
    print("All CMIDs found in the database.")

Detected Windows path. Converting to compatible format...
All CMIDs found in the database.


In [23]:
db_query = """
    unwind $rows as row
    match (m:DATASET {CMID: row.mergingID})-[rs:MERGING]->(s:DATASET {CMID: row.stackID})-[rm:MERGING]->(v:VARIABLE)<-[ru:USES]-(d:DATASET {CMID: row.datasetID})
    return 
    m.CMID as mergingID, m.CMName as mergingName, s.CMID as stackID, s.CMName as stackName, d.CMID as datasetID, d.CMName as datasetName, rs.aggBy as aggBy, v.CMID as variableCMID, head(apoc.coll.flatten(collect(rm.varName),true)) as varName, rm.transform as transform, rm.Rtransform as Rtransform, rm.Rfunction as Rfunction, rm.summaryStatistic as summaryStatistic, ru.Key as Key
    """
data = getQuery(db_query, driver=driver, params={
                "rows": template.to_dict(orient='records')}, type="df")

In [24]:
pd.set_option('display.max_rows', None)

print(data.head(10)) 

  mergingID                   mergingName stackID  \
0      SD55  Hruschka et al. 2015 (Women)  SD2178   
1      SD55  Hruschka et al. 2015 (Women)  SD2178   
2      SD55  Hruschka et al. 2015 (Women)  SD2178   
3      SD55  Hruschka et al. 2015 (Women)  SD2178   
4      SD55  Hruschka et al. 2015 (Women)  SD2178   
5      SD55  Hruschka et al. 2015 (Women)  SD2178   
6      SD55  Hruschka et al. 2015 (Women)  SD2178   
7      SD55  Hruschka et al. 2015 (Women)  SD2178   
8      SD55  Hruschka et al. 2015 (Women)  SD2178   
9      SD55  Hruschka et al. 2015 (Women)  SD2178   

                                stackName datasetID  \
0  Hruschka et al. 2015 (Women) DHS Stack      SD61   
1  Hruschka et al. 2015 (Women) DHS Stack      SD61   
2  Hruschka et al. 2015 (Women) DHS Stack      SD61   
3  Hruschka et al. 2015 (Women) DHS Stack      SD61   
4  Hruschka et al. 2015 (Women) DHS Stack      SD61   
5  Hruschka et al. 2015 (Women) DHS Stack      SD61   
6  Hruschka et al. 2015 (Women)

In [25]:
if "transform" in data.columns:
    print("transforming variables")

    if "Rtransform" in data.columns:
        data['transform'] = data['Rtransform'].combine_first(
            data['transform'])
        data = transform_variables_r(data)

transforming variables


In [26]:
data = data.reset_index()
categories = data[["datasetID","Key"]].copy()
categories = categories.drop_duplicates()
categories = extract_key(categories, col="Key")
categories = categories.melt(
    id_vars=["datasetID",'Key'],
    var_name='variable',
    value_name='value'
)
print(categories.head(10))

  datasetID                                       Key  variable  \
0      SD61                            variable: v404  variable   
1      SD61                            variable: v012  variable   
2      SD61                            variable: v438  variable   
3      SD61  variable: concat(v001,"_",v002,"_",v003)  variable   
4      SD61                            variable: v106  variable   
5      SD61                            variable: v213  variable   
6      SD61                            variable: v437  variable   
7      SD61                        variable: WLTHINDF  variable   
8      SD61                            variable: v102  variable   
9      SD64                            variable: v404  variable   

                            value  
0                            v404  
1                            v012  
2                            v438  
3  concat(v001,"_",v002,"_",v003)  
4                            v106  
5                            v213  
6         

In [27]:

data = pd.merge(data, categories, on=["datasetID","Key"], how="left")
data["variable"] = data["variable"].str.lower()
data = data.astype(str)
data.replace("None", np.nan, inplace=True)

/tmp/ipykernel_1917774/3153981980.py:4: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data.replace("None", np.nan, inplace=True)


In [ ]:
data = pd.merge(data, template[["datasetID", "filePath"]], on="datasetID", how="left")


In [34]:
print(dirpath)
data.to_excel(os.path.join(dirpath, "data.xlsx"), index=False)

./tmp


In [30]:
print(data.head(25)) 

   index mergingID                   mergingName stackID  \
0      0      SD55  Hruschka et al. 2015 (Women)  SD2178   
1      1      SD55  Hruschka et al. 2015 (Women)  SD2178   
2      2      SD55  Hruschka et al. 2015 (Women)  SD2178   
3      3      SD55  Hruschka et al. 2015 (Women)  SD2178   
4      4      SD55  Hruschka et al. 2015 (Women)  SD2178   
5      5      SD55  Hruschka et al. 2015 (Women)  SD2178   
6      6      SD55  Hruschka et al. 2015 (Women)  SD2178   
7      7      SD55  Hruschka et al. 2015 (Women)  SD2178   
8      8      SD55  Hruschka et al. 2015 (Women)  SD2178   
9      9      SD55  Hruschka et al. 2015 (Women)  SD2178   
10    10      SD55  Hruschka et al. 2015 (Women)  SD2178   
11    11      SD55  Hruschka et al. 2015 (Women)  SD2178   
12    12      SD55  Hruschka et al. 2015 (Women)  SD2178   
13    13      SD55  Hruschka et al. 2015 (Women)  SD2178   
14    14      SD55  Hruschka et al. 2015 (Women)  SD2178   
15    15      SD55  Hruschka et al. 2015